In [3]:
!pip install plotly

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 24.4 MB/s eta 0:00:0000:0100:01


In [4]:
!pip install "anywidget>=0.9.13"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.7/213.7 kB 2.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 477.3/477.3 kB 7.1 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 2.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 11.8 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 22.6 MB/s eta 0:00:0000:0100:01


In [5]:
!pip install -U kaleido

In [1]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy import sparse 
from scipy import cluster
import seaborn as sns
import random
import sklearn
from scipy.stats import poisson
from sklearn.neighbors import KernelDensity
import time
import dill
from scipy.optimize import minimize
import pickle
import itertools
import os
import plotly.express as px

In [2]:
df_NN = pd.DataFrame(columns = ['org','celltype','mapping','frac cells'])

In [3]:
a = 0

In [4]:
sm = load_samap('Active_SAMap_Joined/sm_nonneural_hypothalamus_06082026.pk')

In [36]:
org = 'cj'
ref = 'mg'

In [37]:
for item in sm.sams[org].adata.obs['ss_subclass_nounlabeled_nmm_v3_nn'].unique():
    print(item)

327 Oligo NN
326 OPC NN
319 Astro-TE NN
318 Astro-NT NN
339 astrocyte-like NN
341 macrophages NN
Unlabeled
323 Ependymal NN
322 Tanycyte NN
330 VLMC NN
329 ABC NN
325 CHOR NN
340 vascular cells NN
333 Endo NN


In [38]:
mappings = {'317 Astro-CB NN':'Astrocytes',
'318 Astro-NT NN':'Astrocytes',
'319 Astro-TE NN':'Astrocytes',
'320 Astro-OLF NN':'Astrocytes',
'321 Astroependymal NN':'Astrocytes',
'339 astrocyte-like NN':'Astrocytes',
'334 Microglia NN':'Macrophage',
'335 BAM NN':'Macrophage',
'341 macrophages NN':'Macrophage'}

In [39]:
fin_org = []
for item in sm.sams[org].adata.obs['ss_subclass_nounlabeled_nmm_v3_nn']:
    if item in mappings:
        fin_org.append(mappings[item])
    else:
        fin_org.append(item)
        
fin_ref = []
for item in sm.sams[ref].adata.obs['subclass_id_label']:
    if item in mappings:
        fin_ref.append(mappings[item])
    else:
        fin_ref.append(item)

In [40]:
sm.sams[ref].adata.obs['figure2e_mapping'] = fin_ref
sm.sams[org].adata.obs['figure2e_mapping'] = fin_org

In [41]:
ref_level = 'figure2e_mapping'
org_level = 'figure2e_mapping'

In [42]:
keys = {ref:ref_level,org:org_level}
D,MappingTable = get_mapping_scores(sm,keys)

lim_MappingTable = MappingTable.filter(like=org)
lim_MappingTable = lim_MappingTable[lim_MappingTable.index.str.contains(ref)]

/scratch/miniconda/lib/python3.7/site-packages/samap/analysis.py:1609: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.  To get a de-fragmented frame, use `newframe = frame.copy()`
  samap.adata.obs[l] = pd.Categorical(cl)


In [43]:
sam = SAM()
sam.load_data('Active_SAM_joined/SAM_CJ_joined_v2_cleaned_03122025.h5ad')

In [44]:
for item in sm.sams[ref].adata.obs[ref_level].unique():
    print(org + '_' + item)
    if org + '_' + item in lim_MappingTable.columns:
        df_NN.loc[a, 'mapping'] = lim_MappingTable.loc[ref + '_' + item, org + '_' + item]
        df_NN.loc[a, 'org'] = org
        df_NN.loc[a, 'celltype'] = item
        df_NN.loc[a, 'num cells'] = len(sm.sams[org].adata[sm.sams[org].adata.obs[org_level] == item])
        df_NN.loc[a, 'frac cells'] = len(sm.sams[org].adata[sm.sams[org].adata.obs[org_level] == item])/len(sam.adata)
        a += 1

cj_Astrocytes
cj_322 Tanycyte NN
cj_316 Bergmann NN
cj_325 CHOR NN
cj_330 VLMC NN
cj_323 Ependymal NN
cj_324 Hypendymal NN
cj_326 OPC NN
cj_327 Oligo NN
cj_329 ABC NN
cj_328 OEC NN
cj_331 Peri NN
cj_332 SMC NN
cj_333 Endo NN
cj_338 Lymphoid NN
cj_Macrophage
cj_337 DC NN
cj_336 Monocytes NN


In [45]:
df_NN

,org,celltype,mapping,frac cells,num cells
0,dr,Astrocytes,0.455964,0.029319,2424.0
1,dr,322 Tanycyte NN,0.233479,0.001318,109.0
2,dr,316 Bergmann NN,0.324959,0.004971,411.0
3,dr,330 VLMC NN,0.824805,0.005491,454.0
4,dr,323 Ependymal NN,0.778237,0.010499,868.0
5,dr,326 OPC NN,0.844728,0.01859,1537.0
6,dr,327 Oligo NN,0.783695,0.008055,666.0
7,dr,329 ABC NN,0.743983,0.002552,211.0
8,dr,328 OEC NN,0.3341,0.007511,621.0
9,dr,331 Peri NN,0.752488,0.001838,152.0


In [46]:
ct_int = ['327 Oligo NN','326 OPC NN','322 Tanycyte NN','Astrocytes','Macrophage']

In [47]:
df_NN = df_NN[df_NN['celltype'].isin(ct_int)]

In [48]:
df_NN['frac cells'] = df_NN['frac cells'].clip(upper=.1)

/scratch/miniconda/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.


In [49]:
df_NN['frac cells'] = df_NN['frac cells'].astype('float')
df_NN['mapping'] = df_NN['mapping'].astype('float')

/scratch/miniconda/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.
/scratch/miniconda/lib/python3.7/site-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


In [50]:
['322 Tanycyte NN','Astrocytes','Macrophage','326 OPC NN','327 Oligo NN']

['322 Tanycyte NN', 'Astrocytes', 'Macrophage', '326 OPC NN', '327 Oligo NN']

In [51]:
fig = px.scatter(df_NN, x = 'org', y = 'celltype', size = 'frac cells', color = 'mapping', color_continuous_scale= 'Blues', range_color=[0,.99],opacity = 1)
fig.update_xaxes(categoryorder='array', categoryarray= ['cj','ac','xt','dr'])
fig.update_yaxes(categoryorder='array', categoryarray= ['322 Tanycyte NN','Astrocytes','Macrophage','326 OPC NN','327 Oligo NN'])
fig.update_layout(
    autosize=False,
    width=400,
    height=300,
)

fig.write_image("Figures/Figures_06082026/NN_mapping_allorgs_epen_06082026.png")
fig.write_image("Figures/Figures_06082026/NN_mapping_allorgs_epen_06082026.svg")

In [75]:
df_NN

,org,celltype,mapping,frac cells
0,ac,322 Tanycyte NN,0.826871,0.007829
1,ac,319 Astro-TE NN,0.888687,0.023215
2,ac,326 OPC NN,0.948104,0.024113
3,ac,327 Oligo NN,0.952918,0.100000
9,ac,334 Microglia NN,0.802459,0.010898
10,cj,322 Tanycyte NN,0.663972,0.007477
11,cj,319 Astro-TE NN,0.369639,0.006233
15,cj,326 OPC NN,0.969029,0.030659
16,cj,327 Oligo NN,0.877350,0.100000
21,cj,334 Microglia NN,0.866150,0.003545
